In [3]:
# UMUD Challenge: Muscle Architecture in Ultrasound Data
# Objective: To estimate the skeletal muscle architecture, namely pennal agnle (PA), fascicle length (FL) and muscle thickness (MT) from ultrasound images

In [1]:
# import packages
import numpy as np
import pandas as pd
import torch
import torch.nn as nn   
import torch.nn.functional as F
from torchvision import transforms, datasets
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os
from skimage import data, io, filters

In [2]:
# import datasets
from PIL import Image
import cv2

shape = (256, 256)  # Resize images to 256x256

# create directory
base_dir = "/Users/helen1/Desktop/Kaggle CaseComp/UMUD Muscle Architecture_Sep26/apo_training_v1"
images_dir = os.path.join(base_dir, "apo_images_new_model_v1")
masks_dir = os.path.join(base_dir, "apo_masks_new_model_v1")

# get sorted list of files names
filenames = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.tif', '.tiff'))])

images = []
masks = []

for fname in filenames:
    img_path = os.path.join(images_dir, fname)
    mask_path = os.path.join(masks_dir, fname)

    if not os.path.exists(mask_path):
        print(f"Warning: no matching mask for {fname}, skipping.")
        continue

    img = np.array(Image.open(img_path))
    # make sure images are always 1 channel gray scale
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = np.array(Image.open(mask_path))

    images.append(img)
    masks.append(mask)

images = [cv2.resize(img, shape) for img in images]
masks = [cv2.resize(mask, shape, interpolation=cv2.INTER_NEAREST) for mask in masks]

print(f"Loaded {len(images)} image/mask pairs.")

# loaded 1047 image/mask pairs



error: OpenCV(5.0.0) /Users/xperience/GHA-OpenCV-Python/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.simd_helpers.hpp:92: error: (-15:Bad number of channels) in function 'cv::impl::(anonymous namespace)::CvtHelper<cv::impl::(anonymous namespace)::Set<3, 4>, cv::impl::(anonymous namespace)::Set<1>, cv::impl::(anonymous namespace)::Set<0, 2, 5>>::CvtHelper(InputArray, OutputArray, int) [VScn = cv::impl::(anonymous namespace)::Set<3, 4>, VDcn = cv::impl::(anonymous namespace)::Set<1>, VDepth = cv::impl::(anonymous namespace)::Set<0, 2, 5>, sizePolicy = cv::impl::(anonymous namespace)::NONE]'
> Invalid number of channels in input image:
>     'VScn::contains(scn)'
> where
>     'scn' is 1


In [ ]:
import matplotlib.pyplot as plt

def show_pairs(images, masks, n=5, seed=None):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(images), size=min(n, len(images)), replace=False)

    fig, axes = plt.subplots(len(idxs), 3, figsize=(10, 3.3 * len(idxs)))
    axes = np.atleast_2d(axes)

    for row, i in enumerate(idxs):
        img, mask = images[i], masks[i]

        axes[row, 0].imshow(img, cmap="gray" if img.ndim == 2 else None)
        axes[row, 0].set_title(f"Image #{i}")

        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 1].set_title(f"Mask #{i}")

        # Overlay: mask drawn semi-transparently on top of the image
        axes[row, 2].imshow(img, cmap="gray" if img.ndim == 2 else None)
        axes[row, 2].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.5)
        axes[row, 2].set_title("Overlay")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_pairs(images, masks, n=5)

In [ ]:
# Build the U-net model, so we could train a model to predict masks from the images
# reference code: https://www.kaggle.com/code/yousseftaoudi/ultrasound-biomedical-image-segmentation-u-net
# The architecture of this U-net consists of a contracting path to capture context and a symmetric expanding path that enables precise localisation

from tensorflow.keras import datasets, layers, models
from tensorflow.keras.layers import BatchNormalization, Input,UpSampling2D,Concatenate, MaxPooling2D,ReLU, Conv2D,Concatenate,Conv2DTranspose,Cropping2D,Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import SGD

class U_Net():
    def __init__(self, inp=(572, 572, 3), min_filter=64, max_filter=1024, optimizer='Adam',loss=BinaryCrossentropy(), metrics = ['accuracy',AUC()]):
        self.max_filter = max_filter
        self.min_filter = min_filter
        self.img_inp = Input(shape = inp)
        down,cac,down_list = self.downsample()
        lower,down_list = self.contracting_path(down,down_list)
        upper,cac2 = self.upsample(lower,down_list)
        out = Conv2D(1,(1), padding = "same", activation="sigmoid", name='output')(upper)
        self.out = out
        self.model = keras.Model(inputs=self.img_inp, outputs=self.out)
        
        optimizer = tf.keras.optimizers.SGD(
            learning_rate=0.01, momentum=0.9,  name="SGD"
        )
        self.model.compile(optimizer=optimizer,loss=loss,metrics=metrics)

    def downsample(self):
        k = self.min_filter

        down_list = []
        res = self.img_inp
        while k < self.max_filter:
            res = Conv2D(k,(3,3), padding = "same")(res)
            res = ReLU()(res)
            #rest = Dropout(0.3)(res)
            res = Conv2D(k,(3,3), padding = "same")(res)
            res = ReLU()(res)
            #rest = Dropout(0.3)(res)
            down_list.append(res)
            cac = res
            res = MaxPooling2D(pool_size=(2, 2),strides=2)(res)
            curr = res
            k*=2
        return res,cac,down_list    

    def upsample(self,res,down_list):
        k = self.max_filter
        i=0
        while k > self.min_filter:
            res = Conv2D(k//2,(3,3), padding = "same")(res)
            res = ReLU()(res)
            #rest = Dropout(0.3)(res)
            k//=2
            
            if k > self.min_filter:
                res = Conv2D(k//2,(3,3), padding = "same")(res)
                res = ReLU()(res)
                #rest = Dropout(0.3)(res)
                res = UpSampling2D((2,2))(res)
                left = down_list.pop()
                res = self.crop_and_conc(left,res)
                i+=1
            else:
                res = Conv2D(k,(3,3), padding = "same")(res)
                res = ReLU()(res)
                #rest = Dropout(0.3)(res)
            cac = res
            curr = res
        return res,cac

    def contracting_path(self,down,down_list):
        res = Conv2D(self.max_filter,(3,3), padding = "same")(down)
        res = ReLU()(res)
        #rest = Dropout(0.3)(res)
        res = Conv2D(self.max_filter//2,(3,3), padding = "same")(res)
        res = ReLU()(res)
        #rest = Dropout(0.3)(res)
        res = UpSampling2D((2,2))(res)
        left = down_list.pop()
        res = self.crop_and_conc(left,res)
        return res,down_list
    
    
    def crop_and_conc(self,left,right):
        left = Cropping2D(cropping=int((left.shape[1]-right.shape[1])/2))(left)
        right = Concatenate()([left,right])
        return right



In [ ]:
# check the model summary and visualize the model architecture
shape = (256,256,3)

model = U_Net(inp=shape)
model.model.summary()
dot_img_file = 'model.png'
tf.keras.utils.plot_model(model.model, to_file=dot_img_file, show_shapes=False)

In [ ]:
# to check whether model is running on GPU or CPU
print(tf.config.list_physical_devices('GPU'))  # empty list = no GPU found

# to see where operations are actually being placed
#tf.debugging.set_log_device_placement(True)\



In [ ]:
# use images and masks to make a tf.data.Dataset object to feed into U-net model for training
# first transform the images and masks into numpy arrays, then create a tf.data.Dataset object from them
images = np.array(images, dtype="float32")
masks = np.array(masks, dtype="float32")

print(images.shape, masks.shape)

# split our 1047 images into training and validation sets, using 80% for training and 20% for validation
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

# now build the tf.data pipeline for both training and validation datasets
batch_size = 64
autotune = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=len(x_train)) # so to randomly order each epoch
    .batch(batch_size)
    .prefetch(autotune)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(batch_size)
    .prefetch(autotune)
)

# verfiy:
for img_batch, mask_batch in train_ds.take(1):
    print("Image batch shape:", img_batch.shape)
    print("Mask batch shape:", mask_batch.shape)

for img_batch, mask_batch in val_ds.take(1):
    print("Image batch shape:", img_batch.shape)
    print("Mask batch shape:", mask_batch.shape)

In [ ]:
# now feed the training and validation datasets into the U-net model for training
model.model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss="binary_crossentropy", 
    metrics=["accuracy", AUC()]
    )

history = model.model.fit(
    train_ds,
    epochs=5, #only set epochs to 5 to reduce run time, for training purposes, epochs should be much higher
    validation_data=val_ds
)

In [ ]:
# import images and masks on fascicle muscle and repeat the above model
